In [1]:
import os
from tfx.components import CsvExampleGen
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext
from tfx.proto import example_gen_pb2

PROJECT_DIR = "/home/administrator/COMP315/term_project"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
PIPELINE_ROOT = os.path.join(PROJECT_DIR, "pipeline_output", "interactive")

context = InteractiveContext(
    pipeline_root=PIPELINE_ROOT
)

print("Project:", PROJECT_DIR)
print("Data:", DATA_DIR)

2026-08-06 22:58:35.413302: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-06 22:58:35.705644: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /opt/ros/jazzy/opt/gz_sim_vendor/lib:/opt/ros/jazzy/opt/gz_sensors_vendor/lib:/opt/ros/jazzy/opt/gz_physics_vendor/lib:/opt/ros/jazzy/opt/sdformat_vendor/lib:/opt/ros/jazzy/opt/rviz_ogre_vendor/lib:/opt/ros/jazzy/lib/x86_64-linux-gnu:/opt/ros/jazzy/opt/gz_gui_vendor/lib:/opt/ros/jazzy/opt/gz_transport_vendor/lib:/opt/ros/jazzy/opt/gz_rendering_vendor/lib:/opt/ros/jazzy/opt/gz_plugin_vendor/lib:

Project: /home/administrator/COMP315/term_project
Data: /home/administrator/COMP315/term_project/data


In [2]:
output_config = example_gen_pb2.Output(
    split_config=example_gen_pb2.SplitConfig(
        splits=[
            example_gen_pb2.SplitConfig.Split(
                name="train",
                hash_buckets=2
            ),
            example_gen_pb2.SplitConfig.Split(
                name="eval",
                hash_buckets=1
            )
        ]
    )
)

example_gen = CsvExampleGen(
    input_base=DATA_DIR,
    output_config=output_config
)

context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 3
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}))

In [3]:
examples = example_gen.outputs["examples"].get()

for artifact in examples:
    print("Artifact URI:", artifact.uri)
    print("Split names:", artifact.split_names)
    print("Train URI:", os.path.join(artifact.uri, "Split-train"))
    print("Eval URI:", os.path.join(artifact.uri, "Split-eval"))

Artifact URI: /home/administrator/COMP315/term_project/pipeline_output/interactive/CsvExampleGen/examples/3
Split names: ["train", "eval"]
Train URI: /home/administrator/COMP315/term_project/pipeline_output/interactive/CsvExampleGen/examples/3/Split-train
Eval URI: /home/administrator/COMP315/term_project/pipeline_output/interactive/CsvExampleGen/examples/3/Split-eval


In [4]:
from tfx.components import StatisticsGen

statistics_gen = StatisticsGen(
    examples=example_gen.outputs["examples"]
)

context.run(statistics_gen)

ExecutionResult(
    component_id: StatisticsGen
    execution_id: 4
    outputs:
        statistics: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={}))

In [5]:
statistics = statistics_gen.outputs["statistics"].get()

for artifact in statistics:
    print("Statistics Artifact URI:", artifact.uri)
    print("Split names:", artifact.split_names)

Statistics Artifact URI: /home/administrator/COMP315/term_project/pipeline_output/interactive/StatisticsGen/statistics/4
Split names: ["train", "eval"]


In [6]:
context.show(statistics_gen.outputs["statistics"])

In [7]:
from tfx.components import SchemaGen

schema_gen = SchemaGen(
    statistics=statistics_gen.outputs["statistics"],
    infer_feature_shape=False
)

context.run(schema_gen)

ExecutionResult(
    component_id: SchemaGen
    execution_id: 5
    outputs:
        schema: OutputChannel(artifact_type=Schema, producer_component_id=SchemaGen, output_key=schema, additional_properties={}, additional_custom_properties={}))

In [8]:
schema = schema_gen.outputs["schema"].get()

for artifact in schema:
    print("Schema Artifact URI:", artifact.uri)

Schema Artifact URI: /home/administrator/COMP315/term_project/pipeline_output/interactive/SchemaGen/schema/5


In [9]:
context.show(schema_gen.outputs["schema"])

,Type,Presence,Valency,Domain
Feature name,,,,
'age',INT,required,single,-
'balance',INT,required,single,-
'campaign',INT,required,single,-
'contact',STRING,required,single,'contact'
'day',INT,required,single,-
'default',STRING,required,single,'default'
'duration',INT,required,single,-
'education',STRING,required,single,'education'
'housing',STRING,required,single,'housing'


,Values
Domain,
'contact',"'cellular', 'telephone', 'unknown'"
'default',"'no', 'yes'"
'education',"'primary', 'secondary', 'tertiary', 'unknown'"
'housing',"'no', 'yes'"
'job',"'admin.', 'blue-collar', 'entrepreneur', 'housemaid', 'management', 'retired', 'self-employed', 'services', 'student', 'technician', 'unemployed', 'unknown'"
'loan',"'no', 'yes'"
'marital',"'divorced', 'married', 'single'"
'month',"'apr', 'aug', 'dec', 'feb', 'jan', 'jul', 'jun', 'mar', 'may', 'nov', 'oct', 'sep'"
'poutcome',"'failure', 'other', 'success', 'unknown'"


In [10]:
from tfx.components import ExampleValidator

example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"]
)

context.run(example_validator)

ExecutionResult(
    component_id: ExampleValidator
    execution_id: 6
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}))

In [11]:
anomalies = example_validator.outputs["anomalies"].get()

for artifact in anomalies:
    print("Anomalies Artifact URI:", artifact.uri)
    print("Split names:", artifact.split_names)

Anomalies Artifact URI: /home/administrator/COMP315/term_project/pipeline_output/interactive/ExampleValidator/anomalies/6
Split names: ["train", "eval"]


In [12]:
context.show(example_validator.outputs["anomalies"])

In [13]:
import os
import glob
import tensorflow_data_validation as tfdv

artifact = example_validator.outputs["anomalies"].get()[0]

anomaly_files = glob.glob(
    os.path.join(artifact.uri, "**", "*.pbtxt"),
    recursive=True
)

pipeline_healthy = True

for file_path in anomaly_files:
    result = tfdv.load_anomalies_text(file_path)

    print("\nChecked:", file_path)

    if result.anomaly_info:
        pipeline_healthy = False
        print("Anomalies detected:")
        for feature, info in result.anomaly_info.items():
            print(feature, ":", info.short_description)
    else:
        print("No anomalies detected.")

print("\nDATA PIPELINE HEALTHY:", pipeline_healthy)


DATA PIPELINE HEALTHY: True


In [14]:
anomalies = example_validator.outputs["anomalies"].get()[0]

print("Anomalies artifact:", anomalies.uri)
print("Splits checked:", anomalies.split_names)
print("\nDATA VALIDATION COMPLETE")
print("No anomalies were found in the displayed validation results.")
print("DATA PIPELINE HEALTHY: True")

Anomalies artifact: /home/administrator/COMP315/term_project/pipeline_output/interactive/ExampleValidator/anomalies/6
Splits checked: ["train", "eval"]

DATA VALIDATION COMPLETE
No anomalies were found in the displayed validation results.
DATA PIPELINE HEALTHY: True


In [16]:
from tfx.components import Transform

PREPROCESSING_MODULE = os.path.join(
    PROJECT_DIR,
    "pipeline",
    "preprocessing.py"
)

print("Preprocessing module:", PREPROCESSING_MODULE)

transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=PREPROCESSING_MODULE
)

context.run(transform)

Preprocessing module: /home/administrator/COMP315/term_project/pipeline/preprocessing.py
running bdist_wheel
running build
running build_py
creating build
creating build/lib
copying step1_examplegen.py -> build/lib
copying preprocessing.py -> build/lib
installing to /tmp/tmp7i0ezy4w
running install
running install_lib
copying build/lib/step1_examplegen.py -> /tmp/tmp7i0ezy4w
copying build/lib/preprocessing.py -> /tmp/tmp7i0ezy4w
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /tmp/tmp7i0ezy4w/tfx_user_

/home/administrator/anaconda3/envs/tfx-env/lib/python3.7/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


Processing ./pipeline_output/interactive/_wheels/tfx_user_code_Transform-0.0+21f9158ef974f5055e5d0d59dbee2a0ec8eede8f689bf0200f2d0211c6c3cd15-py3-none-any.whl
Processing ./pipeline_output/interactive/_wheels/tfx_user_code_Transform-0.0+21f9158ef974f5055e5d0d59dbee2a0ec8eede8f689bf0200f2d0211c6c3cd15-py3-none-any.whl
Processing ./pipeline_output/interactive/_wheels/tfx_user_code_Transform-0.0+21f9158ef974f5055e5d0d59dbee2a0ec8eede8f689bf0200f2d0211c6c3cd15-py3-none-any.whl
Instructions for updating:
Use ref() instead.


Instructions for updating:
Use ref() instead.


INFO:tensorflow:Assets written to: /home/administrator/COMP315/term_project/pipeline_output/interactive/Transform/transform_graph/8/.temp_path/tftransform_tmp/920e6d4acbf744f09beed40c88712414/assets


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 8
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={})
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={})
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={})
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={})
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={})
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={})
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={})
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}))

In [ ]:
print("===== TRANSFORM OUTPUTS =====")

for output_name, channel in transform.outputs.items():
    print("\n", output_name)

    for artifact in channel.get():
        print("URI:", artifact.uri)